# Full-waveform inversion with Siamese and classical misfits

Runs one inversion of the article under the unified protocol: 30 shots, one receiver per surface grid point,
6 s recording sampled at 3 ms and a 5 Hz Ricker source. Each epoch updates the velocity model once per shot
with Adam; the step is halved every 100 epochs. The step and eps of the velocity model are those selected by
the calibration of the article (supplementary Section S2, Table S3).

**Keys**
- `model`: `'over'` (Overthrust, 200 epochs) or `'marm'` (Marmousi2, 300 epochs)
- `init`: `'linear'` (velocity increasing linearly with depth, the initial model of the article); `'smooth'`,
  `'smooth_top'` (smooth above 2 km, linear below) and `'linear_top'` (the reverse) are used only in the test of
  the deepest band of the Marmousi2 model (Section IV-B1)
- `method`: `'l2'`, `'euclidean'`, `'siamesefwi'`, `'siamesefit'` or `'mpbae'`
- `seed`: initialization of the network; the article uses seeds 0 to 4
- `test`: `None`, `'noise'` (white noise of 10 dB per shot) or `'lowcut'` (frequencies below 3 Hz removed),
  the robustness tests of Section V-B (SiameseFit, three seeds, run with the cuDNN autotuner off)

Outputs are written to `results/<model>_<init>[_<test>]/<method>/seed<seed>/`: `config.json`, `metrics.csv`,
`model_final.npy` and checkpoints every 10 epochs, used by the explainability analyses (`XAI.ipynb`).
On an RTX A4000, one run takes 8 to 22 min on the Overthrust model and 15 to 36 min on the Marmousi2 model.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from utils.fwi import run, run_dir, calibrated
from utils.plots import NAMES, plot_models, plot_curves

In [ ]:
model = 'marm'         # 'marm' or 'over'
init = 'linear'        # 'linear' (article); 'smooth', 'smooth_top' or 'linear_top' (deep-band test)
method = 'siamesefit'  # 'l2', 'euclidean', 'siamesefwi', 'siamesefit' or 'mpbae'
seed = 0
test = None            # None, 'noise' or 'lowcut'

lr, eps = calibrated(model, init, method)
options = {'noise': dict(noise_snr=10, cudnn_benchmark=False), 'lowcut': dict(lowcut=3, cudnn_benchmark=False)}.get(test, {})
print('Step of the velocity model: %g m/s, Adam eps: %g' % (lr, eps))


In [ ]:
metrics = run(model, init, method, seed, **options)

In [ ]:
inverted = np.load(run_dir(model, init, method, seed, test) + '/model_final.npy')
plot_models(model, init, inverted, NAMES[method])
plt.show()
plot_curves(metrics, NAMES[method])
plt.show()
print('Final SNR %.2f dB, SSIM %.3f' % (metrics['snr'][-1], metrics['ssim'][-1]))